# درس ۱۱ - پروتکل عامل‌به‌عامل (A2A)


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## پروتکل A2A چیست؟

**پروتکل عامل‌به‌عامل (A2A)** یک استاندارد باز است که به عامل‌های هوش مصنوعی اجازه می‌دهد ارتباط برقرار کنند،
یکدیگر را کشف کنند، و همکاری کنند — حتی وقتی روی فریم‌ورک‌های مختلف ساخته شده‌اند یا توسط
سرویس‌های مختلف میزبانی می‌شوند.

مفاهیم کلیدی:

- **کشف** – عامل‌ها یک *کارت عامل* منتشر می‌کنند که توانایی‌هایشان را توصیف می‌کند، که باعث می‌شود
  آسان‌تر باشد برای سایر عامل‌ها (یا هماهنگ‌کننده‌ها) تا متخصص مناسب را برای یک وظیفه پیدا کنند.
- **ارسال پیام** – عامل‌ها پیام‌های ساخت‌یافته را از طریق یک پروتکل مشترک تبادل می‌کنند، بنابراین یک
  درخواست از یک عامل می‌تواند توسط عامل دیگر درک و برآورده شود بدون توجه به پیاده‌سازی داخلی
  آن.
- **چرخه عمر وظیفه** – A2A حالت‌هایی مانند *ارسال‌شده*، *درحال‌کار*، *تکمیل‌شده* و
  *شکست‌خورده*، به هماهنگ‌کننده دید کامل می‌دهد نسبت به اینکه یک کار واگذار‌شده چگونه پیشرفت می‌کند.

در این درس همکاری به سبک A2A را با اتصال سه عامل تخصصی سفر
به یک جریان کاری شبیه‌سازی می‌کنیم که هر عامل در آن تخصص خود را ارائه می‌دهد و نتایج را به عامل بعدی منتقل می‌کند.


## ایجاد نمایندگان سفر تخصصی


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## همکاری چندعاملی از طریق گردش‌کار

ما سه عامل را به یک گردش‌کار متوالی متصل می‌کنیم که بازتاب‌دهندهٔ ارسال پیام A2A است:

1. **CurrencyExchangeAgent** درخواست کاربر را دریافت می‌کند و راهنمایی ارزی تولید می‌کند.
2. **ActivityPlannerAgent** زمینهٔ غنی‌شده را دریافت می‌کند و پیشنهادهای فعالیت را اضافه می‌کند.
3. **TravelManagerAgent** هر دو ورودی را ترکیب می‌کند و یک خلاصه نهایی سفر تهیه می‌کند.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## درک A2A در محیط تولید

در یک محیط تولیدی پروتکل A2A سناریوهای قدرتمند میان‌سرویسی را امکان‌پذیر می‌سازد:

| قابلیت | توضیحات |
|---|---|
| **هم‌تعاملی میان چارچوب‌ها** | یک عامل که با یک چارچوب ساخته شده می‌تواند وظایف را به عاملی که با هر چارچوب سازگار با A2A ساخته شده واگذار کند، و امکان تعامل واقعی بین سازمان‌ها را فراهم آورد. |
| **مرزهای سرویس** | عوامل می‌توانند در ریزخدمات جداگانه، مناطق ابری یا حتی سازمان‌های مختلف مستقر باشند در حالی که همچنان به‌صورت یکپارچه همکاری می‌کنند. |
| **کشف پویا** | یک ارکستراتور می‌تواند در زمان اجرا از رجیستری Agent Card پرس‌وجو کند تا مناسب‌ترین متخصص را برای یک زیروظیفهٔ مشخص پیدا کند. |
| **استریم و اعلان‌های پوش** | A2A از Server-Sent Events (SSE) برای به‌روزرسانی‌های پیشرفت در زمان واقعی و از اعلان‌های پوش برای وظایف طولانی‌مدت پشتیبانی می‌کند. |

جریان کاری که بالا ساختیم نسخه‌ای ساده‌شده و درون‌پردازشی از این الگو است. در یک استقرار واقعی هر عامل یک نقطه پایانی HTTP را افشا می‌کرد، یک Agent Card منتشر می‌کرد و از طریق پروتکل A2A JSON-RPC ارتباط برقرار می‌نمود.


## خلاصه

در این درس آموختید:

1. **پروتکل A2A چیست** — یک استاندارد باز برای کشف، ارسال پیام،
   و مدیریت وظایف.
2. **چگونه عامل‌های تخصصی ایجاد کنیم** — یک عامل تبدیل ارز، یک عامل برنامه‌ریز فعالیت،
   و یک هماهنگ‌کننده مدیر سفر.
3. **چگونه عامل‌ها را در یک جریان کاری به هم متصل کنیم** — با استفاده از `WorkflowBuilder` برای مدل‌سازی ارسال پیام
   ترتیبی بین عامل‌ها.
4. **چگونه A2A در تولید کار می‌کند** — امکان همکاری بین چارچوب‌ها و سرویس‌ها
   را با کشف پویا و به‌روزرسانی‌های جریان‌دار فراهم می‌کند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نواقصی باشند. سند اصلی به زبان اصلی خود باید به عنوان منبع معتبر تلقی شود. برای اطلاعات حساس یا حیاتی، ترجمهٔ حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
